# 01 — Load Data (Hydraulic Condition Monitoring)

This notebook loads raw sensor `.txt` files and labels (`profile.txt`) **via `utils/data_io.py`**, 
builds the feature matrix `X` and labels `y`, and saves them to `data/processed/`.

In [1]:
from pathlib import Path
import sys

# Make sure Python can import from the repo root (where utils/ lives)
# If this notebook is inside notebooks/, parent[0] is the repo root.
repo_root = Path.cwd()
if not (repo_root / "utils" / "data_io.py").exists():
    repo_root = Path.cwd().parents[0]  # e.g., notebooks/ -> repo root
sys.path.insert(0, str(repo_root))

from utils.data_io import (
    ensure_dirs, load_profile, load_all_sensors,
    build_feature_matrix, save_processed, write_feature_index,
    ROOT_DIR, RAW_DIR, META_DIR, PROC_DIR
)

print("ROOT_DIR :", ROOT_DIR)
print("RAW_DIR  :", RAW_DIR)
print("META_DIR :", META_DIR)
print("PROC_DIR :", PROC_DIR)

ensure_dirs()


ROOT_DIR : C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard
RAW_DIR  : C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\raw
META_DIR : C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\metadata
PROC_DIR : C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\processed


In [2]:
labels_df = load_profile()
labels_df.shape, labels_df.head()

✅ Loaded profile.txt | shape=(2205, 5) | from=data\metadata\profile.txt


((2205, 5),
    0    1  2    3  4
 0  3  100  0  130  1
 1  3  100  0  130  1
 2  3  100  0  130  1
 3  3  100  0  130  1
 4  3  100  0  130  1)

In [3]:
sensor_frames, sensor_order = load_all_sensors()
len(sensor_frames), sensor_order[:5]

— Loading sensor files —
   Loaded CE.txt         | shape=(2205, 60)     | delim=whitespace
   Loaded CP.txt         | shape=(2205, 60)     | delim=whitespace
   Loaded EPS1.txt       | shape=(2205, 6000)   | delim=whitespace
   Loaded FS1.txt        | shape=(2205, 600)    | delim=whitespace
   Loaded FS2.txt        | shape=(2205, 600)    | delim=whitespace
   Loaded PS1.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded PS2.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded PS3.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded PS4.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded PS5.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded PS6.txt        | shape=(2205, 6000)   | delim=whitespace
   Loaded SE.txt         | shape=(2205, 60)     | delim=whitespace
   Loaded TS1.txt        | shape=(2205, 60)     | delim=whitespace
   Loaded TS2.txt        | shape=(2205, 60)     | delim=whitespace
   Loaded TS3.txt        | shape=(220

(17, ['CE', 'CP', 'EPS1', 'FS1', 'FS2'])

In [4]:
X, y = build_feature_matrix(sensor_frames, sensor_order, labels_df)
X.shape, y.shape

✅ Built feature matrix X | shape=(2205, 43680)
✅ Prepared labels y       | shape=(2205, 5) | columns=['cooler_condition', 'valve_condition', 'internal_pump_leakage', 'hydraulic_accumulator', 'stable_flag']


((2205, 43680), (2205, 5))

In [5]:
EXPECTED_ROWS = 2205

row_mismatch = [(k, df.shape) for k, df in sensor_frames.items() if df.shape[0] != EXPECTED_ROWS]
print("Row mismatches:", "none" if not row_mismatch else row_mismatch)

print("\nFirst 5 label rows:")
y.head()


Row mismatches: none

First 5 label rows:


,cooler_condition,valve_condition,internal_pump_leakage,hydraulic_accumulator,stable_flag
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


In [6]:
x_path, y_path = save_processed(X, y)
print("Saved:", x_path, y_path)

✅ Saved X -> data\processed\X_features.parquet
✅ Saved y -> data\processed\y_labels.parquet
Saved: C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\processed\X_features.parquet C:\Users\melny\OneDrive\Desktop\Projects\hydraulic_dashboard\data\processed\y_labels.parquet


In [7]:
meta_idx_path = write_feature_index(X)
meta_idx_path

✅ Wrote metadata -> data\metadata\feature_index.csv; rows=43680


WindowsPath('C:/Users/melny/OneDrive/Desktop/Projects/hydraulic_dashboard/data/metadata/feature_index.csv')